# Brinkman stabilization and Oseen transport

Check affine velocity and pressure under increasing drag, including every residual term in the P1/P1 USFEM form. A separate Oseen patch verifies the skew-advection and Robin sign conventions.

The local provider declares the grad-grad velocity/pressure blocks and the
independent pairings $B_T\lambda=\langle\lambda,v\rangle$ and
$C_T(u,p)=-\langle u,\mu\rangle$. The USFEM contribution is

$$
\begin{aligned}
a_T^{\mathrm{USFEM}}&=a_T^{\mathrm{TH}}
 +\gamma(u,v)_T-\tau_T(R(u,p),R(v,q))_T,\\
R(u,p)&=\gamma u+\nabla p,\qquad
\tau_T=\frac{h_T^2}{\max(\gamma h_T^2,12\nu)+12\nu},\\
L_T(v,q)&=(f,v)_T-\tau_T(f,R(v,q))_T.
\end{aligned}
$$

These are P1/P1 fine-cell forms with constant coefficients and $m=1/3$.
The Oseen control uses P2/P1 and the skew term
$[(\beta\cdot\nabla u,v)-(u,\beta\cdot\nabla v)]/2$.
The multiplier is negative grad-grad pseudotraction, including the half-advection
normal contribution; it is not symmetric Cauchy traction. The global `Equation`
explicitly supplies negative Dirichlet velocity moments and the reconstructed
pressure has zero physical integral. Sources, spaces and gauges match the
original consistency controls.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))


In [ ]:
from functools import partial
import numpy as np
from pymhm import Equation, MultiscaleProblem, assemble, TriangleMesh, SkeletonSpace, FaceSpace
from pymhm.fem.scalar.operators import boundary_data
from examples.formulations.flow import (
    VelocityPressureSpace, velocity_pressure_equations,
    velocity_pressure_fields, p1_residual_weight,
)


In [ ]:
def velocity(x):
    """Return a rigid rotation with zero divergence."""
    return np.column_stack((x[:, 1], -x[:, 0]))

mesh = TriangleMesh.unit_square(2)
space = SkeletonSpace(mesh, tuple(FaceSpace.uniform(1) for _ in mesh.faces), 2)
for drag in (0.0, 1.0, 100.0):
    local_space = VelocityPressureSpace(mesh, space, 1, 1, 4, 5)
    provider = partial(
        velocity_pressure_equations, space=local_space,
        source=lambda x, d=drag: d * velocity(x) + np.array([1.0, 2.0]),
        drag=drag, residual_weight=p1_residual_weight,
    )
    boundary, _ = boundary_data(space, velocity, order=5)
    problem = MultiscaleProblem(
        Equation(0, -np.r_[boundary, np.zeros(2 * len(mesh.cells))]),
        provider, range(len(mesh.cells)), space.size, (2,) * len(mesh.cells),
    )
    system = assemble(problem)
    gauge = system.mean_constraint([record[2] for record in system.local_metadata], 0.0)
    coefficients = system.solve(constraints=(gauge,))
    result = velocity_pressure_fields(system, coefficients, local_space)
    error = result.pressure_l2_error(lambda x: x[:, 0] + 2 * x[:, 1] - 1.5)
    assert result.l2_error(velocity) < 1e-10 and error < 1e-9
    print("drag", drag, "pressure error", error)

# The Oseen control retains the original Taylor–Hood P2/P1 spaces and r=2.
local_space = VelocityPressureSpace(mesh, space, 2, 1, 2, 5)
provider = partial(
    velocity_pressure_equations, space=local_space,
    source=lambda x: velocity(x) + [3.0, -2.0], drag=1.0, advection=(2.0, 3.0),
)
problem = MultiscaleProblem(
    Equation(0, -np.r_[boundary, np.zeros(2 * len(mesh.cells))]),
    provider, range(len(mesh.cells)), space.size, (2,) * len(mesh.cells),
)
system = assemble(problem)
gauge = system.mean_constraint([record[2] for record in system.local_metadata], 0.0)
oseen = velocity_pressure_fields(system, system.solve(constraints=(gauge,)), local_space)
assert oseen.l2_error(velocity) < 1e-10


This is a polynomial consistency check, not a proof of uniform robustness in unresolved Brinkman or convection boundary layers. The native drag and background advection are constant; tensor/variable local coefficients may be expressed through UFL.